In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
OUT_DIR = PROJECT_ROOT / "results" / "data_audits"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def audit_duplicates(path, key_cols):
    df = pd.read_csv(path)
    df["truth_date"] = pd.to_datetime(df["truth_date"], errors="coerce")

    n_rows = len(df)
    n_unique_keys = df[key_cols].drop_duplicates().shape[0]
    n_duplicate_extra_rows = n_rows - n_unique_keys
    n_rows_in_duplicated_keys = df.duplicated(key_cols, keep=False).sum()

    duplicated_rows = (
        df[df.duplicated(key_cols, keep=False)]
        .sort_values(key_cols)
        .copy()
    )

    return {
        "file": path.name,
        "n_rows": n_rows,
        "n_unique_keys": n_unique_keys,
        "n_duplicate_extra_rows": int(n_duplicate_extra_rows),
        "n_rows_in_duplicated_keys": int(n_rows_in_duplicated_keys),
        "key_cols": ", ".join(key_cols)
    }, duplicated_rows


audit_rows = []
duplicated_parts = []

# 1. Historical latest files: disease is implicit in filename
for fname in ["latest-ARI_incidence.csv", "latest-ILI_incidence.csv"]:
    path = DATA_DIR / fname
    if path.exists():
        summary, dup = audit_duplicates(path, ["location", "truth_date"])
        audit_rows.append(summary)
        if len(dup) > 0:
            dup.insert(0, "file", fname)
            duplicated_parts.append(dup)
    else:
        print("Missing:", path)

# 2. Weekly snapshot files: disease/target is either in target column or filename
snapshot_files = sorted(DATA_DIR.glob("20*-*-*_incidence.csv"))

for path in snapshot_files:
    df_head = pd.read_csv(path, nrows=5)
    if "target" in df_head.columns:
        key_cols = ["target", "location", "truth_date"]
    else:
        key_cols = ["location", "truth_date"]

    summary, dup = audit_duplicates(path, key_cols)
    audit_rows.append(summary)
    if len(dup) > 0:
        dup.insert(0, "file", path.name)
        duplicated_parts.append(dup)

audit_df = pd.DataFrame(audit_rows)

if duplicated_parts:
    duplicated_df = pd.concat(duplicated_parts, ignore_index=True)
else:
    duplicated_df = pd.DataFrame()

audit_df.to_csv(OUT_DIR / "raw_data_duplicate_audit_summary.csv", index=False)
duplicated_df.to_csv(OUT_DIR / "raw_data_duplicate_rows.csv", index=False)

display(audit_df)

print("\nTOTAL EXTRA DUPLICATE ROWS:", audit_df["n_duplicate_extra_rows"].sum())
print("TOTAL ROWS IN DUPLICATED KEYS:", audit_df["n_rows_in_duplicated_keys"].sum())

if len(duplicated_df) > 0:
    print("\nDuplicated rows found:")
    display(duplicated_df.head(30))
else:
    print("\nNo duplicated country-week observations found under the checked keys.")

,file,n_rows,n_unique_keys,n_duplicate_extra_rows,n_rows_in_duplicated_keys,key_cols
0,latest-ARI_incidence.csv,6685,6685,0,0,"location, truth_date"
1,latest-ILI_incidence.csv,10646,10646,0,0,"location, truth_date"
2,2024-10-11-ARI_incidence.csv,6662,6662,0,0,"target, location, truth_date"
3,2024-10-11-ILI_incidence.csv,10616,10616,0,0,"target, location, truth_date"
4,2024-10-18-ARI_incidence.csv,6685,6685,0,0,"target, location, truth_date"
...,...,...,...,...,...,...
115,2025-12-05-ILI_incidence.csv,4650,4650,0,0,"target, location, truth_date"
116,2025-12-12-ARI_incidence.csv,3370,3370,0,0,"target, location, truth_date"
117,2025-12-12-ILI_incidence.csv,4671,4671,0,0,"target, location, truth_date"
118,2025-12-19-ARI_incidence.csv,3386,3386,0,0,"target, location, truth_date"



TOTAL EXTRA DUPLICATE ROWS: 0
TOTAL ROWS IN DUPLICATED KEYS: 0

No duplicated country-week observations found under the checked keys.
